# 02 - Response baseline

A Response LightGBM classifier predicting `P(conversion | X)`, ignoring
treatment entirely.

This is **not** a causal estimator. It answers "who is likely to convert,"
not "who converts *because of* the treatment." It is the non-causal
targeting comparator the uplift models are measured against -- a strong
response model that ranks poorly on uplift is exactly the point.

Run `01_data_processing.ipynb` first.

In [ ]:
import sys
from pathlib import Path


def _find_repo_root() -> Path:
    """Locate the repo root without assuming the working directory.

    A fresh Kaggle kernel starts in /kaggle/working, not in the repo, so we
    search: the current directory and its parents (local development), then
    /kaggle/working and each attached /kaggle/input/<slug>/ (Kaggle, where
    the repo is cloned into working or attached as a dataset).
    """
    bases = [Path.cwd(), *Path.cwd().parents, Path("/kaggle/working"), Path("/kaggle/input")]
    for base in bases:
        if not base.is_dir():
            continue
        if (base / "src" / "data.py").is_file():
            return base
        for child in sorted(p for p in base.iterdir() if p.is_dir()):
            if (child / "src" / "data.py").is_file():
                return child
    raise RuntimeError(
        "Could not locate the repository root (no src/data.py found). On Kaggle, "
        "clone this repository into /kaggle/working or attach it as a dataset."
    )


REPO_ROOT = _find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print("repo root:", REPO_ROOT)

In [ ]:
# Kaggle's stock image does not include econml (and may not match the
# pinned lightgbm version). Installing quietly here is a no-op if the pinned
# versions are already present -- e.g. local dev via requirements.txt.
%pip install -q econml==0.17.0 lightgbm==4.7.0
import econml
import lightgbm
print("econml:", econml.__version__, " lightgbm:", lightgbm.__version__)

In [ ]:
from src.data import PRIMARY_OUTCOME, TREATMENT_COLUMN, load_config, load_parquet, output_dir, save_parquet
from src.preprocessing import LightGBMFeatureTransform
from src.models import fit_response_model, predict
from src.evaluation import evaluate_ranking, response_diagnostics

CONFIG = load_config()
OUTPUT_DIR = output_dir()

train_frame = load_parquet(OUTPUT_DIR / "train.parquet")
val_frame = load_parquet(OUTPUT_DIR / "validation.parquet")
test_frame = load_parquet(OUTPUT_DIR / "test.parquet")
train_frame.shape, val_frame.shape, test_frame.shape

## Feature representation

Continuous features stay `float64`; the eight categorical features become a
**train-fitted** pandas categorical dtype, so LightGBM uses native
categorical splits rather than treating the token as an ordered number. The
vocabulary is fit on train only and reused unchanged on validation and test.

In [ ]:
transform = LightGBMFeatureTransform().fit(train_frame)
X_train, X_val, X_test = (transform.transform(f) for f in (train_frame, val_frame, test_frame))

Y_train, Y_val, Y_test = (f[PRIMARY_OUTCOME] for f in (train_frame, val_frame, test_frame))
T_val, T_test = val_frame[TREATMENT_COLUMN], test_frame[TREATMENT_COLUMN]
X_train.dtypes.value_counts()

## Fit

In [ ]:
response_model = fit_response_model(X_train, Y_train, X_val, Y_val, seed=CONFIG["seed"])
print("best iteration:", response_model.best_iteration)

## Evaluate (validation)

Two deliberately different questions:

1. **Response diagnostics** -- does it predict conversion well? (AUC/AP/logloss)
2. **Uplift ranking** -- does ranking by predicted response probability also
   rank well by *incremental* conversion?

A high AUC with a weak Qini/AUUC is the expected, informative result: response
probability is not a causal score.

In [ ]:
val_scores = predict(response_model, X_val)
response_diagnostics(val_scores, Y_val)

In [ ]:
val_ranking = evaluate_ranking(val_scores, T_val, Y_val)
print("validation qini_above_random:", round(val_ranking.qini_above_random, 4))
print("validation auuc_above_random:", round(val_ranking.auuc_above_random, 4))
val_ranking.uplift_at_k

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(val_ranking.qini_curve["coverage"], val_ranking.qini_curve["qini_gain"], label="Response model")
ax1.plot([0, 1], [0, val_ranking.theoretical_random_qini_area * 2], "--", label="Theoretical random")
ax1.set_xlabel("Population coverage"); ax1.set_ylabel("Qini gain"); ax1.set_title("Qini curve"); ax1.legend()

ax2.plot(val_ranking.uplift_curve["coverage"], val_ranking.uplift_curve["uplift_gain"], label="Response model")
ax2.plot([0, 1], [0, val_ranking.theoretical_random_auuc_area * 2], "--", label="Theoretical random")
ax2.set_xlabel("Population coverage"); ax2.set_ylabel("Uplift gain"); ax2.set_title("Uplift curve (AUUC)"); ax2.legend()
fig.tight_layout()

## Save test predictions

Scored once and written out, so notebook 04's final comparison can evaluate
every model on the same untouched test partition without refitting anything.

In [ ]:
import pandas as pd

test_predictions = pd.DataFrame({
    "score": predict(response_model, X_test),
    TREATMENT_COLUMN: T_test.to_numpy(),
    PRIMARY_OUTCOME: Y_test.to_numpy(),
})
save_parquet(test_predictions, OUTPUT_DIR / "preds_response_test.parquet")
print("saved ->", OUTPUT_DIR / "preds_response_test.parquet")

## Next

Continue with `03_uplift_models.ipynb`.